## Make a sick banner for the repo

In [21]:
import h5py
import numpy as np
import cmasher as cmr
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap as LSC

from embers_passes import PassFile

prd = LSC.from_list("pride", cmr.pride(np.linspace(0.1, 0.8, 256)))
prd.set_under("white")

In [3]:
pf = PassFile("../passes/rf0/S06XX_rf0XX_passes.h5")
passes = pf.read_passes(pointing=0)

In [32]:
outdir = Path("frames")
outdir.mkdir(exist_ok=True)

slices =  [1, 2, 4, 8, 16, 32, 48, 64, 96, 128, 192, 256, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664, 1792, 1823]

for i, s in enumerate(slices):

    all_alt = np.concatenate([p.alt_deg for p in passes[:s]])
    all_az = np.concatenate([p.az_deg for p in passes[:s]])
    all_power = np.concatenate([p.power_db for p in passes[:s]])

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.hexbin(
        all_az,
        all_alt,
        C=all_power,
        reduce_C_function=np.mean,
        gridsize=64,
        cmap=prd,
        vmax=0,
        vmin=-64
    )

    ax.set_xlim(0, 360)
    ax.set_ylim(7, 81)

    # --- remove all axes clutter ---
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # optional: remove any margins inside axes
    ax.margins(0)

    # --- transparent backgrounds ---
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    # --- save tightly cropped ---
    outfile = outdir / f"frame_{i:03d}.png"
    fig.savefig(
        outfile,
        dpi=150,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close(fig)